In [1]:
print(1)

1


# Step 1

In [2]:
import json
import shutil
from pathlib import Path

import anndata as ad
import numpy as np
import os
import pandas as pd

In [3]:
from tqdm import tqdm

In [4]:
def filter_samples(adata):
    adata = adata.copy()
    perturbagen_mask = ~adata.obs['pubchem_cid'].isna()
    return adata[perturbagen_mask].copy()

In [6]:
def construct_df(adata):
    adata = adata.copy()

    dataset = adata.obs['dataset'].to_numpy()
    context = adata.obs['cell_type'].to_numpy()
    perturbations = adata.obs['pubchem_cid'].to_numpy()
    dose = adata.obs['pert_dose_uM'].to_numpy()
    time = adata.obs['pert_time_h'].to_numpy()

    values = adata.layers[LAYER]
    readout_names = adata.var.index.to_numpy()

    n_perts, n_readouts = values.shape

    df = pd.DataFrame({
    'dataset':  np.repeat(dataset, n_readouts),
    'context': np.repeat(context, n_readouts),
    'perturbation': np.repeat(perturbations, n_readouts),
    'log_dose': np.log10(np.repeat(dose, n_readouts)),
    'time': np.repeat(time, n_readouts),
    'readout': np.tile(readout_names, n_perts),
    'value': values.ravel(),
    })
    return df

In [7]:
def filter_readout(df):
    df = df.copy()

    na_mask = df.isna().any(axis=1)
    #duplicates_mask = df[['dataset', 'context', 'perturbation', 'dose', 'time', 'readout']].duplicated()
    readouts_mask = na_mask
    
    if df[readouts_mask].shape[0] != 0:
        print(f'Missing values: {df[readouts_mask].shape[0]}')
        
    #readouts_mask = readouts_mask | duplicates_mask
    #if df[readouts_mask].shape[0] != 0:
    #    print(f'Duplicates: {df[readouts_mask].shape[0]}')
    
    df = df[~readouts_mask].copy()

    return df

In [8]:
def run_construct_df(adata):
    
    adata = filter_samples(adata)
    df = construct_df(adata)
    df = filter_readout(df)

    return df

In [9]:
LAYER = 'logFC'
PLIBDATA_ROOT = '/home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets'
BASE_PATH = '/lustre/groups/ml01/workspace/olga.novitskaia/data_updated'

#deg_folders = os.listdir(BASE_PATH)

In [10]:
dict_paths = {
'tahoe': f'{BASE_PATH}/tahoe/deg_data/group_rep/full/qc_false/filter_min_cells_50/results/',
'l1000_phase1': f'{BASE_PATH}/l1000_phase1/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',
'l1000_phase2': f'{BASE_PATH}/l1000_phase2/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',
'novartis': f'{BASE_PATH}/novartis_batch_2500/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',

'vcpi_0001': f'{BASE_PATH}/vcpi_0001/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',
'vcpi_0002': f'{BASE_PATH}/vcpi_0002/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',
'gdpx2': f'{BASE_PATH}/gdpx2/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',

'cigs_mce': f'{BASE_PATH}/cigs_mce/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',
'cigs_tcm': f'{BASE_PATH}/cigs_tcm/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',

'dili_train': f'{BASE_PATH}/dilimap_train/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',

}

In [11]:
for i, key in enumerate(dict_paths.keys()):
    print(f'{i} out of {len(dict_paths.keys())}')
    for j, file in enumerate(os.listdir(dict_paths[key])):
        print(f'{j} out of {len(os.listdir(dict_paths[key]))}')
        path = dict_paths[key] + file
        adata = ad.read_h5ad(path)
        df = run_construct_df(adata)
        
        os.makedirs(f'{PLIBDATA_ROOT}/{key}/', exist_ok=True)
        df.to_parquet(f'{PLIBDATA_ROOT}/{key}/{file.split('_de')[0]}.parquet')
        #break

0 out of 10
0 out of 48
1 out of 48
2 out of 48
3 out of 48
4 out of 48
5 out of 48
6 out of 48
7 out of 48
8 out of 48
9 out of 48
10 out of 48
11 out of 48
12 out of 48
13 out of 48
14 out of 48
15 out of 48
16 out of 48
17 out of 48
18 out of 48
19 out of 48
20 out of 48
21 out of 48
22 out of 48
23 out of 48
24 out of 48
25 out of 48
26 out of 48
27 out of 48
28 out of 48
29 out of 48
30 out of 48
31 out of 48
32 out of 48
33 out of 48
34 out of 48
35 out of 48
36 out of 48
37 out of 48
38 out of 48
39 out of 48
40 out of 48
41 out of 48
42 out of 48
43 out of 48
44 out of 48
45 out of 48
46 out of 48
47 out of 48
1 out of 10
0 out of 70


/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


1 out of 70
2 out of 70
3 out of 70
4 out of 70
5 out of 70


/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


6 out of 70
7 out of 70


...

Missing values: 546265
1 out of 2


/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Missing values: 458655
9 out of 10
0 out of 1


/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Missing values: 1169269
